In [1]:
from google.colab import userdata
import os

# --- 1. Define constants and secrets ---
GIT_TOKEN = userdata.get('github_token')
GITHUB_USER = 'yguo005'
GITHUB_REPO = 'medgemma_chatbot'
BRANCH_NAME = 'main'

# --- 2. ALWAYS start from a clean state in /content ---
# Go back to the root content directory to avoid nested paths
%cd /content

# Remove the repository directory if it already exists to ensure a fresh clone
!rm -rf {GITHUB_REPO}

# --- 3. clone the branch ---
!git clone https://github.com/yguo005/medgemma_chatbot.git

# --- 4. Change directory into the newly cloned project ---
%cd {GITHUB_REPO}

/content
Cloning into 'medgemma_chatbot'...
remote: Enumerating objects: 329, done.
remote: Counting objects: 100% (329/329), done.
remote: Compressing objects: 100% (210/210), done.
remote: Total 329 (delta 151), reused 273 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (329/329), 22.49 MiB | 13.19 MiB/s, done.
Resolving deltas: 100% (151/151), done.
/content/medgemma_chatbot


In [2]:
# ---Install System and Python Dependencies ---
print("Installing all packages with latest compatible versions...")
!pip install -q \
    torch \
    "transformers>=4.42.4" \
    accelerate \
    bitsandbytes \
    langchain \
    langchain-community \
    langchain-openai \
    faiss-cpu \
    fastapi \
    uvicorn \
    python-multipart \
    pypdf \
    python-dotenv \
    google-cloud-aiplatform \
    pyngrok \
    pydantic \
    starlette
!pip install --upgrade "pydantic>=2.0.0"
!pip install --upgrade langchain langchain-community langchain-openai
!pip install --upgrade torch torchvision torchaudio
!pip install -U bitsandbytes

print(" \nInstallation completed!")

Installing all packages with latest compatible versions...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
 
Installation completed!


In [3]:
from google.colab import auth, userdata
import os
from huggingface_hub import login

# Authenticate for Google Cloud services
auth.authenticate_user()

# Set environment variables from Colab Secrets
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['NGROK_AUTHTOKEN'] = userdata.get('NGROK_AUTHTOKEN')

# Manually set other env vars for the demo
os.environ['DEPLOYMENT_MODE'] = 'development'
os.environ['USE_MEDGEMMA_GARDEN'] = 'false'

#  Log in to Hugging Face
# This uses the HF_TOKEN secret to authenticate session
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

In [4]:
# Build the Knowledge Base
!python /content/medgemma_chatbot/src/services/ai/rag/create_memory_for_llm.py


 Configuration Status:
   Mode: development
   AI Service Mode: hybrid
   MedGemma: Local HF
   Valid: True

 Creating FAISS Vector Store for AI Health Consultant
 Current script: /content/medgemma_chatbot/src/services/ai/rag/create_memory_for_llm.py
 Project root: /content/medgemma_chatbot
 Data path: /content/medgemma_chatbot/data/document
 FAISS path: /content/medgemma_chatbot/data/vectorstore/db_faiss
 Project root exists: True
 Data directory exists: True

 Loaded 759 documents from 1 PDF file(s)
 Created 7080 text chunks.
 OpenAI Embedding Model Loaded (Vector Dimension: 1536)
 FAISS vector store already exists. Overwriting...
 FAISS vector store created and saved to: /content/medgemma_chatbot/data/vectorstore/db_faiss


In [6]:
# Run the FastAPI Server using ngrok
import os
import sys
import asyncio
from pyngrok import ngrok, conf

sys.path.insert(0, os.path.abspath('src'))


# Validate environment variables
NGROK_TOKEN = os.environ.get("NGROK_AUTHTOKEN")
OPENAI_KEY = os.environ.get("OPENAI_API_KEY")

if not NGROK_TOKEN:
    print(" ERROR: NGROK_AUTHTOKEN not set!")
    print("Set it with: os.environ['NGROK_AUTHTOKEN'] = 'your-token-here'")
    sys.exit(1)

if not OPENAI_KEY:
    print(" ERROR: OPENAI_API_KEY not set!")
    print("Set it with: os.environ['OPENAI_API_KEY'] = 'your-key-here'")
    sys.exit(1)


# Set the ngrok auth token
conf.get_default().auth_token = NGROK_TOKEN

async def run_fastapi():
    try:
        # Use nest_asyncio to allow uvicorn to run in a notebook
        import nest_asyncio
        nest_asyncio.apply()

        # Import uvicorn
        import uvicorn

        print(" Starting FastAPI server...")
        print(f" Working directory: {os.getcwd()}")

        # Check if main.py exists
        if not os.path.exists("main.py"):
            print(" ERROR: main.py not found in current directory!")
            print("Make sure you're in the correct directory with your FastAPI app.")
            return

        # Configure uvicorn server
        config = uvicorn.Config(
            "main:app",
            host="0.0.0.0",
            port=8000,
            log_level="info",
            reload=False  # Disable reload in Colab
        )
        server = uvicorn.Server(config)

        # Open a tunnel to the uvicorn server
        print(" Opening ngrok tunnel...")
        public_url = ngrok.connect(8000)
        print(f" FastAPI server is live at: {public_url}")
        print(f" Mobile interface: {public_url}/mobile.html")
        print(f" Desktop interface: {public_url}/")
        print(f" API docs: {public_url}/docs")
        print("\n To stop the server, interrupt this cell (Runtime > Interrupt execution)")

        # Run the server
        await server.serve()

    except Exception as e:
        print(f" Error starting server: {e}")
        import traceback
        traceback.print_exc()
    finally:
        # Clean up ngrok tunnels
        try:
            ngrok.disconnect(8000)
            print(" Cleaned up ngrok tunnel")
        except:
            pass

# Run the server asynchronously
await run_fastapi()

 Starting FastAPI server...
 Working directory: /content/medgemma_chatbot
 Opening ngrok tunnel...
 FastAPI server is live at: NgrokTunnel: "https://9aa8f68967ef.ngrok-free.app" -> "http://localhost:8000"
 Mobile interface: NgrokTunnel: "https://9aa8f68967ef.ngrok-free.app" -> "http://localhost:8000"/mobile.html
 Desktop interface: NgrokTunnel: "https://9aa8f68967ef.ngrok-free.app" -> "http://localhost:8000"/
 API docs: NgrokTunnel: "https://9aa8f68967ef.ngrok-free.app" -> "http://localhost:8000"/docs

 To stop the server, interrupt this cell (Runtime > Interrupt execution)

 Configuration Status:
   Mode: development
   AI Service Mode: hybrid
   MedGemma: Local HF
   Valid: True



INFO:     Started server process [178]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     73.92.81.141:0 - "GET / HTTP/1.1" 200 OK
INFO:     73.92.81.141:0 - "GET /static/css/style.css HTTP/1.1" 200 OK
INFO:     73.92.81.141:0 - "GET /static/js/main.js HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [178]


 Cleaned up ngrok tunnel


In [ ]:
# Debug: Test the conversation flow before starting the server
import sys
sys.path.insert(0, os.path.abspath('src'))

try:
    # Test imports
    from src.services.ai.rag.chatbot import Chatbot
    from src.services.conversation.manager import ConversationManager
    from src.services.ai.ai_service_manager import create_ai_service_manager
    from src.services.safety.safety_guardrails import MedicalSafetyGuardrails

    print(" All imports successful")

    # Test service initialization
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

    # Initialize services
    chatbot = Chatbot(
        openai_api_key=OPENAI_API_KEY,
        use_medgemma_garden=False,
        gcp_project_id=None,
        endpoint_id=None
    )

    ai_service_manager = create_ai_service_manager("hybrid")

    conversation_manager = ConversationManager(
        ai_service=ai_service_manager,
        rag_service=chatbot
    )

    print(" Services initialized successfully")

    # Test the conversation flow
    import asyncio

    async def test_conversation():
        try:
            response = await conversation_manager.process_message("test_session", "i have headache", False)
            print(" Conversation test successful:")
            print(f"   Response type: {response.get('response_type')}")
            print(f"   Response: {response.get('response_text', response.get('response', 'No response'))[:100]}...")
            return True
        except Exception as e:
            print(" Conversation test failed:")
            print(f"   Error: {e}")
            import traceback
            traceback.print_exc()
            return False

    # Run the test
    success = await test_conversation()

    if success:
        print("\n All tests passed! The server should work correctly.")
    else:
        print("\n  There are issues that need to be fixed before starting the server.")

except Exception as e:
    print(f" Setup failed: {e}")
    import traceback
    traceback.print_exc()

 All imports successful
 Services initialized successfully


config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]